<a href="https://colab.research.google.com/github/AlexGalindo08/proyecto-imagenes-medicas-pdi/blob/main/Pia%20segmentaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# SECCIÓN 3.3: SEGMENTACIÓN POR UMBRAL Y TRANSFORMADA DE FOURIER
# =====================================================================


from ipywidgets import interact, IntSlider


def segmentacion_interactiva(valor_umbral):
    """
    Función interactiva que aplica umbralización fija global y adaptativa,
    permitiendo ajustar el umbral en tiempo real con un slider.
    """
    # 1. Aplicación de Umbral Fijo Global (Usa el valor dinámico del slider)
    _, thresh_fijo = cv2.threshold(gris, valor_umbral, 255, cv2.THRESH_BINARY)

    # 2. Aplicación de Umbral Adaptativo Local (Ponderación Gaussiana)
    thresh_adapt = cv2.adaptiveThreshold(
        gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )

    # Despliegue de los resultados para comparación visual directa
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(gris, cmap='gray')
    plt.title("Radiografía Original")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(thresh_fijo, cmap='gray')
    plt.title(f"Umbral Fijo Global (T={valor_umbral})")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(thresh_adapt, cmap='gray')
    plt.title("Umbral Adaptativo Gaussiano")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Ejecución del control interactivo en Colab
print("Ajuste el valor de 'valor_umbral' para interactuar con la segmentación:")
interact(segmentacion_interactiva, valor_umbral=IntSlider(min=0, max=255, step=1, value=125));


# =====================================================================
# ANÁLISIS EN FRECUENCIA (TRANSFORMADA DE FOURIER 2D)
# =====================================================================

# 1. Obtener la FFT 2D y desplazar las bajas frecuencias al centro
f_transform = np.fft.fft2(gris)
f_shift = np.fft.fftshift(f_transform)

# 2. Calcular el espectro de magnitud en escala logarítmica para visualización
espectro_magnitud = 20 * np.log(np.abs(f_shift) + 1)

# 3. Diseño del Filtro Pasa-Bajas Ideal (Máscara circular)
filas, columnas = gris.shape
centro_f, centro_c = filas // 2, columnas // 2
radio_corte = 40  # Radio en píxeles para atenuar alta frecuencia

# Creamos la máscara binaria del tamaño de la imagen
mascara_pasa_bajas = np.zeros((filas, columnas), np.uint8)
cv2.circle(mascara_pasa_bajas, (centro_c, centro_f), radio_corte, 1, -1)

# 4. Multiplicación del espectro por la máscara circular (Filtrado)
f_shift_filtrado = f_shift * mascara_pasa_bajas

# 5. Deshacer el centrado (shift) y calcular la Transformada Inversa (IFFT)
f_ishift = np.fft.ifftshift(f_shift_filtrado)
img_reconstruida = np.fft.ifft2(f_ishift)
img_reconstruida = np.abs(img_reconstruida)

# =====================================================================
# VISUALIZACIÓN DE RESULTADOS DE FOURIER
# =====================================================================
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(gris, cmap='gray')
plt.title("Original (Espacio)")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(espectro_magnitud, cmap='gray')
plt.title("Espectro de Magnitud FFT 2D")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_reconstruida, cmap='gray')
plt.title(f"Reconstruida (Pasa-Bajas, R={radio_corte})")
plt.axis('off')

plt.tight_layout()
plt.show()

print("Etapa de procesamiento en frecuencia concluida.")